# Generate `config.json` for the Pipeline

This notebook will:
1. Search through the chosen folders to gather the Anchor and Target Maps.
2. Create a dictionary (`config`) containing all required paths.
3. Write this configuration to `input/config.json` for use in `run_pipeline.py`.


N.B. Anchor Maps are the maps that we want to georeference, while Target Maps are the ones that are already georeferenced and that we'd like to use as reference.



## 1. Imports and Initial Setup

In this first cell, we import any necessary libraries and define our folder structure (where to look for Anchor and Target Maps). 

For this example, we will use the following folder structure:
```
input_maps/
    anchor_maps/
    target_maps/
```

With each folder containg the maps we want to use for the pipeline as folders.

In [1]:
import os
import json

# Define where anchor and target maps are located
anchor_dir = "input_maps/anchor"
target_dir = "input_maps/target"

# Create an "input" folder if it doesn't exist, since we'll write config there
os.makedirs('input', exist_ok=True)


## 2. Collect Anchor Map Files

We loop through each subfolder in `anchor_dir`. For anchor maps, we expect:
- A `.jpeg`,`.jpg`,`.png`,`.tif` or `.tiff` image,
- A `_mask.png` mask,
- A `.points` file containing georeferencing information.

We'll collect the full paths to these files in lists so that we can later build our config. In the folder, we provide three maps as examples to test the pipeline.


In [2]:
anchor_image_paths = []
anchor_mask_paths = []
anchor_points_paths = []

# Loop through subfolders in the anchor directory
for anchor_map_name in os.listdir(anchor_dir):
    anchor_map_folder = os.path.join(anchor_dir, anchor_map_name)
    if not os.path.isdir(anchor_map_folder):
        continue  # Skip non-folders

    # Search for the needed files
    # EXCLUDE anything with '_mask' from the main image search
    image_file = [
        f for f in os.listdir(anchor_map_folder)
        if f.lower().endswith((".jpeg", ".jpg", ".png", ".tif", ".tiff"))
        and "_mask" not in f.lower()  # <--- important fix
    ]
    mask_file  = [
        f for f in os.listdir(anchor_map_folder)
        if f.lower().endswith("_mask.png")
    ]
    points_file = [
        f for f in os.listdir(anchor_map_folder)
        if f.lower().endswith(".points")
    ]

    # Make sure we have at least one match for each required file
    if image_file and mask_file and points_file:
        anchor_image_paths.append(os.path.join(anchor_map_folder, image_file[0]))
        anchor_mask_paths.append(os.path.join(anchor_map_folder, mask_file[0]))
        anchor_points_paths.append(os.path.join(anchor_map_folder, points_file[0]))

# Quick check
print("Collected anchor images: ", anchor_image_paths)
print("Collected anchor masks:  ", anchor_mask_paths)
print("Collected anchor points: ", anchor_points_paths)



Collected anchor images:  ['input_maps/anchor/1860_pierotti/pierotti_1860.jpeg', 'input_maps/anchor/1865_wilson/wilson_1865.jpeg', 'input_maps/anchor/1846_vandevelde/vandevelde_1846.png']
Collected anchor masks:   ['input_maps/anchor/1860_pierotti/pierotti_1860_mask.png', 'input_maps/anchor/1865_wilson/wilson_1865_mask.png', 'input_maps/anchor/1846_vandevelde/vandevelde_1846_mask.png']
Collected anchor points:  ['input_maps/anchor/1860_pierotti/pierotti_1860.points', 'input_maps/anchor/1865_wilson/wilson_1865.points', 'input_maps/anchor/1846_vandevelde/vandevelde_1846.points']


## 3. Collect Target Map Files

We do a similar process for target maps. However, we do **not** expect (or require) a `.points` file. We'll also create a subfolder named `propagated_gcp_results` for each target map to store pipeline outputs.

In our case, we still provided our manual georeferencing files, which can be used by the user to compare the results of the pipeline with the manual georeferencing. In the folder, we provide one map to test the pipeline.

In [3]:
target_image_paths = []
target_mask_paths = []
output_dirs = []

# Loop through subfolders in the target directory
for target_map_name in os.listdir(target_dir):
    target_map_folder = os.path.join(target_dir, target_map_name)
    if not os.path.isdir(target_map_folder):
        continue  # Skip non-folders

    # Search for the needed files
    image_file = [f for f in os.listdir(target_map_folder) if f.lower().endswith((".jpeg", ".jpg"))]
    mask_file  = [f for f in os.listdir(target_map_folder) if f.endswith("_mask.png")]

    # Make sure we have at least one match for each required file
    if image_file and mask_file:
        full_image_path = os.path.join(target_map_folder, image_file[0])
        full_mask_path = os.path.join(target_map_folder, mask_file[0])
        
        target_image_paths.append(full_image_path)
        target_mask_paths.append(full_mask_path)
        
        # Create an output directory where pipeline results will be saved
        output_dir = os.path.join(target_map_folder, 'propagated_gcp_results')
        os.makedirs(output_dir, exist_ok=True)
        output_dirs.append(output_dir)

# Quick check
print("Collected target images: ", target_image_paths)
print("Collected target masks:  ", target_mask_paths)
print("Output directories:      ", output_dirs)


Collected target images:  ['input_maps/target/1860_bartholomew/bartholomew_1860.jpeg']
Collected target masks:   ['input_maps/target/1860_bartholomew/bartholomew_1860_mask.png']
Output directories:       ['input_maps/target/1860_bartholomew/propagated_gcp_results']


## 4. Build and Save the `config.json`

Finally, we assemble our dictionary for the pipeline and save it out to `input/config.json`.


In [4]:
config = {
    "target_image_paths": target_image_paths,
    "target_image_mask_paths": target_mask_paths,
    "anchor_image_paths": anchor_image_paths,
    "anchor_image_masks": anchor_mask_paths,
    "anchor_points_paths": anchor_points_paths,
    "output_dirs": output_dirs
}

config_path = 'input/config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)

print(f"Configuration saved to {config_path}")
print(json.dumps(config, indent=4))


Configuration saved to input/config.json
{
    "target_image_paths": [
        "input_maps/target/1860_bartholomew/bartholomew_1860.jpeg"
    ],
    "target_image_mask_paths": [
        "input_maps/target/1860_bartholomew/bartholomew_1860_mask.png"
    ],
    "anchor_image_paths": [
        "input_maps/anchor/1860_pierotti/pierotti_1860.jpeg",
        "input_maps/anchor/1865_wilson/wilson_1865.jpeg",
        "input_maps/anchor/1846_vandevelde/vandevelde_1846.png"
    ],
    "anchor_image_masks": [
        "input_maps/anchor/1860_pierotti/pierotti_1860_mask.png",
        "input_maps/anchor/1865_wilson/wilson_1865_mask.png",
        "input_maps/anchor/1846_vandevelde/vandevelde_1846_mask.png"
    ],
    "anchor_points_paths": [
        "input_maps/anchor/1860_pierotti/pierotti_1860.points",
        "input_maps/anchor/1865_wilson/wilson_1865.points",
        "input_maps/anchor/1846_vandevelde/vandevelde_1846.points"
    ],
    "output_dirs": [
        "input_maps/target/1860_bartholomew/p

## Performing the georeferencing

Now you can use run_pipeline.py to run the pipeline with the configuration file generated here either in the terminal or directly in the next cell. Beware that depending on the amount of maps to georeference and on the georeferenced maps that are available the process can take a long time.

```bash
python scripts/run_pipeline.py input/config.json
```

```python


In [5]:
!python scripts/run_pipeline.py input/config.json

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loaded SuperPoint model
Loaded SuperGlue model ("outdoor" weights)
Loaded SuperGlue model ("outdoor" weights)
Processing anchor maps: 0it [00:00, ?it/s]INFO:modules.MapDataset:Image size is within limits.
INFO:modules.MapDataset:[DEBUG] For map: pierotti_1860.jpeg
INFO:modules.MapDataset:  Image shape: (7812, 11862, 3) (dtype: uint8)
INFO:modules.MapDataset:Loaded mask from input_maps/anchor/1860_pierotti/pierotti_1860_mask.png with shape (7812, 11862)
INFO:modules.MapDataset:Mask stats -> Non-zero pixels: 60876368 / 92665944 (65.69%)
INFO:modules.MapDataset:  SuperPoint detected 990 keypoints before mask filtering for map: pierotti_1860.jpeg
INFO:modules.MapDataset:I